# Identifier drift case studies (audit + explainability)

This notebook generates *manuscript-ready* qualitative evidence for IDTrack's core framing: identifier mapping is a reproducibility contract across **namespace × release × assembly**.

It automatically mines **case studies** from the IDTrack graph (using `explain=True`) and exports:

- Figure: `idtrack-manuscript/figures/fig_drift_case_studies.pdf`
- Table (optional): `idtrack-manuscript/tables/drift_case_studies.tex`

Caching:
- Case-study payloads (including explainability paths) are cached under `idtrack/docs/_notebooks/idtrack_cache/experiments/case_studies/`.
- If the cache is missing, the notebook computes it (graph loading can be memory-intensive).

## Interpretation guide (how to use these in the Results text)

- Each panel shows a single query identifier time-travelled into a target release and projected into a target namespace.
- Drift is not treated as "noise": a merge/split/retirement is a *historical curation event* that legitimately changes identity.
- The audit path is the marketing artifact: it turns a mapping into a rerunnable, inspectable claim rather than a black-box lookup.


In [ ]:
from __future__ import annotations

import time
from pathlib import Path

import pandas as pd

import matplotlib.pyplot as plt

try:
    import networkx as nx
except Exception as e:  # noqa: S110
    nx = None
    print('Warning: networkx not available:', e)

import sys

# Add experiments/src to sys.path
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not ((REPO_ROOT / 'idtrack').is_dir() and (REPO_ROOT / 'idtrack-manuscript').is_dir()):
    REPO_ROOT = REPO_ROOT.parent

EXPERIMENTS_SRC = REPO_ROOT / 'idtrack' / 'reproducibility' / 'experiments' / 'src'
sys.path.append(str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    MANUSCRIPT_COLORS,
    apply_rcparams,
    atomic_write_text,
    experiments_cache_dir,
    idtrack_cache_dir,
    load_rcparams,
    manuscript_figures_dir,
    manuscript_tables_dir,
    read_pickle,
    write_pickle,
)

try:
    apply_rcparams(load_rcparams())
except Exception as e:  # noqa: S110
    print('Warning: could not apply shared rcParams:', e)

plt.rcParams.update({'savefig.dpi': 300, 'figure.dpi': 140})

IDTRACK_LOCAL_REPO = idtrack_cache_dir(REPO_ROOT)
CACHE_DIR = experiments_cache_dir(REPO_ROOT, experiment='case_studies')
MANUSCRIPT_FIGURES = manuscript_figures_dir(REPO_ROOT)
MANUSCRIPT_TABLES = manuscript_tables_dir(REPO_ROOT)

print('Repo root:', REPO_ROOT)
print('IDTRACK_LOCAL_REPO:', IDTRACK_LOCAL_REPO)
print('CACHE_DIR:', CACHE_DIR)


In [ ]:
# -------------------- Configuration --------------------

ORGANISM_ALIAS = 'human'
GRAPH_SNAPSHOT_RELEASE = 114
TO_RELEASE = 107
FINAL_DATABASE = 'HGNC Symbol'
STRATEGY = 'all'

# Mining settings
N_CANDIDATES_TO_SCAN = 1500
N_CASES = 4
RANDOM_SEED = 0

CASES_PKL = CACHE_DIR / (
    f"cases_{ORGANISM_ALIAS}_snapshot{GRAPH_SNAPSHOT_RELEASE}_to{TO_RELEASE}_final{FINAL_DATABASE}_"
    f"strategy{STRATEGY}_n{N_CASES}_seed{RANDOM_SEED}.pickle"
)

print('CASES_PKL:', CASES_PKL)


In [ ]:
# -------------------- Mine case studies (cache-first) --------------------

import numpy as np

if CASES_PKL.exists():
    cases = read_pickle(CASES_PKL)
    print('Loaded:', CASES_PKL)
else:
    import idtrack

    rng = np.random.default_rng(RANDOM_SEED)

    api = idtrack.API(local_repository=str(IDTRACK_LOCAL_REPO))
    api.configure_logger()

    organism, latest = api.resolve_organism(ORGANISM_ALIAS)
    snapshot = max(int(GRAPH_SNAPSHOT_RELEASE), int(TO_RELEASE))
    if snapshot > int(latest):
        raise ValueError(f'snapshot_release={snapshot} exceeds latest={latest} for {ORGANISM_ALIAS}')

    print(f'Building/loading graph: {organism} snapshot_release={snapshot}')
    api.build_graph(organism_name=organism, snapshot_release=snapshot, calculate_caches=True)

    g = api.track.graph

    # Reservoir-sample ENSG identifiers without materializing the full list
    def reservoir_sample(prefix: str, k: int) -> list[str]:
        sample: list[str] = []
        n_seen = 0
        for node in g.nodes:
            if not isinstance(node, str) or not node.startswith(prefix):
                continue
            n_seen += 1
            if len(sample) < k:
                sample.append(node)
                continue
            j = int(rng.integers(0, n_seen))
            if j < k:
                sample[j] = node
        return sample

    candidates = reservoir_sample('ENSG', N_CANDIDATES_TO_SCAN)
    print('Sampled candidates:', len(candidates))

    def pick_release_for_ensembl(node: str) -> int | None:
        try:
            ranges = g.get_active_ranges_of_id[node]
        except Exception:
            return None
        if not ranges:
            return None
        lo, hi = ranges[int(rng.integers(0, len(ranges)))]
        hi_int = int(max(g.graph.get('confident_for_release', [snapshot]))) if hi == float('inf') else int(hi)
        if lo > hi_int:
            return None
        return int(rng.integers(int(lo), hi_int + 1))

    cases = []
    t0 = time.perf_counter()
    for q in candidates:
        if len(cases) >= N_CASES:
            break
        fr = pick_release_for_ensembl(q)
        if fr is None:
            continue

        res = api.convert_identifier(
            q,
            from_release=fr,
            to_release=int(TO_RELEASE),
            final_database=FINAL_DATABASE,
            strategy=STRATEGY,
            explain=True,
        )

        if res.get('no_corresponding') or res.get('no_conversion'):
            continue

        n_targets = len(res.get('target_id', []) or [])
        changed = (n_targets == 1 and res['target_id'][0] != res.get('query_id'))

        if n_targets > 1 or changed:
            cases.append({'from_release': fr, 'result': res})

    dt = time.perf_counter() - t0
    print(f'Collected {len(cases)} cases in {dt:.1f}s')

    write_pickle(cases, CASES_PKL)
    print('Saved:', CASES_PKL)

cases


In [ ]:
# -------------------- Summarize + export table --------------------

rows = []
for i, case in enumerate(cases, start=1):
    res = case['result']
    rows.append(
        {
            'case': i,
            'query_id': res.get('query_id'),
            'from_release': int(case.get('from_release')),
            'to_release': int(TO_RELEASE),
            'final_database': res.get('final_database'),
            'n_targets': len(res.get('target_id', []) or []),
            'targets': ', '.join(map(str, (res.get('target_id') or [])[:6])),
        }
    )

summary = pd.DataFrame(rows)
summary

# Optional LaTeX export (compact)
out_tex = MANUSCRIPT_TABLES / 'drift_case_studies.tex'

if summary.empty:
    print('No cases available; skipping LaTeX export.')
else:
    lines = []
    lines.append(r'\begin{table}[t]')
    lines.append(r'\caption{Identifier drift case studies mined from the IDTrack graph (audit-friendly, explainable conversions).}')
    lines.append(r'\label{tab:drift_cases}')
    lines.append(r'\centering')
    lines.append(r'\footnotesize')
    lines.append(r'\begin{tabular}{@{}r l r r r@{}}')
    lines.append(r'\toprule')
    lines.append(r'Case & Query & From & To & |Targets| \\')
    lines.append(r'\midrule')

    for r in summary.itertuples(index=False):
        lines.append(
            f"{r.case} & \\texttt{{{r.query_id}}} & {r.from_release} & {r.to_release} & {r.n_targets} " + r"\\"
        )

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')

    atomic_write_text(out_tex, '\n'.join(lines) + '\n')
    print('Wrote:', out_tex)


In [ ]:
# -------------------- Plot case-study paths --------------------

import numpy as np

if not cases:
    print('No cases to plot.')
elif nx is None:
    print('networkx not available; skipping path plots.')
else:
    def node_kind(n) -> str:
        if n is None:
            return 'none'
        if isinstance(n, int):
            return 'bridge'
        s = str(n)
        if s.startswith('ENS'):
            return 'ensembl'
        return 'external'

    kind_color = {
        'ensembl': MANUSCRIPT_COLORS['1→1'],
        'external': MANUSCRIPT_COLORS['1→n'],
        'bridge': MANUSCRIPT_COLORS['neutral'],
        'none': '#FFFFFF',
    }

    n = min(len(cases), N_CASES)
    ncols = 2
    nrows = (n + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4.5 * nrows), constrained_layout=True)
    axes = list(np.ravel(axes))

    for ax, case in zip(axes, cases[:n]):
        res = case['result']
        paths = res.get('the_path', {})
        if not paths:
            ax.axis('off')
            ax.set_title(f"{res.get('query_id')} (no path)")
            continue

        # Plot the first available target path
        key = next(iter(paths.keys()))
        path_steps = paths[key]

        edges = []
        nodes_in_order = []
        for step in path_steps:
            if not isinstance(step, (list, tuple)) or len(step) < 2:
                continue
            u, v = step[0], step[1]
            if u is None or v is None:
                continue
            edges.append((u, v))
            nodes_in_order.extend([u, v])

        G = nx.DiGraph()
        G.add_edges_from(edges)

        # Simple left-to-right layout following the observed order
        uniq = []
        seen = set()
        for v in nodes_in_order:
            if v in seen:
                continue
            seen.add(v)
            uniq.append(v)

        pos = {v: (i, 0.0) for i, v in enumerate(uniq)}

        node_colors = [kind_color.get(node_kind(v), '#CCCCCC') for v in G.nodes]
        nx.draw_networkx_edges(G, pos=pos, ax=ax, arrows=True, arrowstyle='-|>', arrowsize=12, width=1.5)
        nx.draw_networkx_nodes(G, pos=pos, ax=ax, node_size=650, node_color=node_colors, linewidths=0.5, edgecolors='#333333')

        labels = {v: (str(v) if len(str(v)) <= 18 else str(v)[:18] + '…') for v in G.nodes}
        nx.draw_networkx_labels(G, pos=pos, ax=ax, labels=labels, font_size=8)

        ax.set_axis_off()
        ax.set_title(f"{res.get('query_id')} r{case.get('from_release')} → r{TO_RELEASE} ({FINAL_DATABASE})")

    for ax in axes[n:]:
        ax.axis('off')

    out_fig = MANUSCRIPT_FIGURES / 'fig_drift_case_studies.pdf'
    fig.savefig(out_fig, bbox_inches='tight')
    print('Saved:', out_fig)
